<dl class="meta-row">
<dt>Data</dt><dd>Net short position disclosures under EU Regulation 236/2012, published by national regulators in the EU, the UK, Norway and Iceland</dd>
<dt>Sample</dt><dd>19 countries, position dates through September 2026 (update of September 28, 2026)</dd>
<dt>Code</dt><dd>Scraping and cleaning pipeline in two notebooks; the Norway download is the example below</dd>
<dt>Author</dt><dd>Qingyi (Freda) Song Drechsler</dd>
</dl>

<div class="btn-row">
<a class="primary" href="../notebooks/eu_short_norway.ipynb" download>Download Norway example (.ipynb)</a>
</div>



## policy Background

Following the global financial crisis of 2008, European Securities and Markets Authority (ESMA) launched policy EU N236/2012 on short selling and certain aspects of credit default swaps. The purpose of this EU-wide reporting regulation is to increase market stability by reducing the opaqueness of market activities through mandatory daily reporting by institutional investors.

Therefore, each participating country's financial monitoring agency requires market participants with the significant short positions on qualifying securities to self-report the positions, and further makes such information available to the public.

## technical background

As each government agency's website carries the data differently, and often the data scraping exercise involves mimicking the "clicking" action of a mouse, the package Selenium suits the need perfectly.

However, I would like to emphasize here that this code needs human monitoring over time, as government agency's website can change layout or the presented data might show different format. So please do not rely on the code output blindly (more likely it will crash somewhere before getting to the end!).

Our student intern Shuai Zheng contributed to this code.

This code was revamped significantly to reflect the selenium package update as well as changes in the host countries' websites.

Update: October 2026

## the data

The tables and chart below summarize the combined dataset as of the latest update (September 28, 2026): every
net short position disclosure published by the 19 regulators covered, with the position holder,
the company (issuer and ISIN), the net short position as a percentage of issued share capital,
and the position date.

In [2]:
#| column: page
#| echo: false
# coverage by country
cov = (eu.groupby('country')
         .agg(disclosures=('position_date', 'size'), companies=('isin', 'nunique'),
              holders=('position_holder', 'nunique'),
              first=('position_date', 'min'), last=('position_date', 'max'))
         .sort_values('disclosures', ascending=False))
total = pd.DataFrame({'disclosures': [len(eu)], 'companies': [eu['isin'].nunique()],
                      'holders': [eu.position_holder.nunique()],
                      'first': [eu.position_date.min()], 'last': [eu.position_date.max()]},
                     index=['All countries'])
cov = pd.concat([cov, total])
out = pd.DataFrame({
    'Disclosures': cov.disclosures.map('{:,}'.format),
    'Companies (ISINs)': cov.companies.map('{:,}'.format),
    'Position holders': cov.holders.map('{:,}'.format),
    'First position date': cov['first'].dt.strftime('%Y-%m-%d'),
    'Last position date': cov['last'].dt.strftime('%Y-%m-%d'),
}, index=cov.index.rename('Country'))
out

,Disclosures,Companies (ISINs),Position holders,First position date,Last position date
Country,,,,,
UK,"112,510",941,621,2012-10-31,2026-07-09
Belgium,"81,689",63,192,2012-10-31,2026-09-16
Germany,"61,498","1,038",489,2003-03-26,2026-09-21
France,"47,941",271,530,2012-10-03,2026-09-18
Sweden,"39,732",465,486,2010-05-10,2026-09-21
Italy,"25,705",232,307,2012-08-30,2026-09-21
Netherlands,"24,705",99,281,2012-11-01,2026-09-21
Spain,"13,743",90,254,2010-06-10,2026-09-18
Finland,"10,939",80,239,2012-10-31,2026-09-21


The chart counts the companies with at least one disclosure dated in each month, which is
comparable across countries; raw disclosure counts are not, since some regulators (Belgium until
2022, for example) repeat unchanged positions every day.

In [3]:
#| echo: false
# companies with a disclosed position each month: top 4 countries + the rest
eu['month'] = eu.position_date.dt.to_period('M').dt.to_timestamp()
# Nov 2012 (regulation in force) to Jun 2026 (UK and Denmark files end in July)
m = eu.loc[(eu.month >= '2012-11-01') & (eu.month <= '2026-06-01')]
by = m.drop_duplicates(['country', 'month', 'isin']).groupby(['month', 'country']).size()
by = by.unstack(fill_value=0)
top = by.sum().sort_values(ascending=False).index[:4]
series = [(c, by[c], color) for c, color in
          zip(top, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100'])]
series.append(('All other countries', by.drop(columns=top).sum(axis=1), '#8a8984'))

fig = go.Figure()
for name, s, color in series:
    fig.add_trace(go.Scatter(x=s.index, y=s, name=name, mode='lines',
                             line=dict(color=color, width=2),
                             hovertemplate='%{y:,}<extra>' + name + '</extra>'))
fig.update_layout(
    title=dict(text='Companies with a Disclosed Net Short Position, by Month'
                    '<br><sup style="color:#52514e">Distinct ISINs with at least one '
                    'position disclosure dated in the month, '
                    f'{by.index[0]:%b %Y}\u2013{by.index[-1]:%b %Y}</sup>',
               x=0, xanchor='left'),
    template='plotly_white', height=500, margin=dict(l=60, r=30, t=90, b=110),
    font=dict(family='Inter, system-ui, sans-serif', size=13, color='#0b0b0b'),
    paper_bgcolor='#fcfcfb', plot_bgcolor='#fcfcfb', hovermode='x unified',
    legend=dict(orientation='h', x=0, y=-0.12, yanchor='top'),
    xaxis=dict(showgrid=False, ticks='outside', tickcolor='#c3c2b7',
               linecolor='#c3c2b7', hoverformat='%b %Y'),
    yaxis=dict(title='Companies', gridcolor='#ecebe7', zeroline=False, rangemode='tozero'),
)
fig.show()

## the code

The code runs in two parts. Part 1 downloads each regulator's raw disclosure file, using Selenium
for sites that need clicking and plain HTTP requests where a file or API address is available.
Part 2 reads each country's file, standardizes the columns, runs validation checks, and appends
the new records to the previous update, since some regulators do not keep their full history
online.

The example below is the Norway download, which reads the regulator's API directly instead of
clicking through a website.

### example: Norway

Norwegian Finanstilsynet presents the data in the json format through its API portal after the most recent website revamp, therefore the code does not resemble the other countries' download process.

In [1]:
# import packages
import json

import certifi
import pandas as pd
import urllib3

In [2]:
http = urllib3.PoolManager(cert_reqs='CERT_REQUIRED',
                           ca_certs=certifi.where())
url = "https://ssr.finanstilsynet.no/api/v2/instruments"
r = http.request('GET', url)
assert r.status == 200, f"Norway API returned status {r.status}"

data = json.loads(r.data.decode('utf-8'))
df = pd.json_normalize(data, 'events', ["isin", "issuerName"],
                       max_level=0)
dfValid = df.loc[df.shares != 0]
print(f"Norway: {df.shape[0]} raw records, "
      f"{dfValid.shape[0]} with a nonzero position")

Norway: 3364 raw records, 3187 with a nonzero position


In [3]:
#| column: page
df.head()

,date,shortPercent,shares,activePositions,isin,issuerName
0,2026-10-01T00:00:00,0.00,0,[],BMG9156K1018,2020 BULKERS
1,2026-09-30T00:00:00,1.80,413635,"[{'date': '2026-09-30T00:00:00', 'shortPercent...",BMG9156K1018,2020 BULKERS
2,2026-09-29T00:00:00,1.64,377008,"[{'date': '2026-09-29T00:00:00', 'shortPercent...",BMG9156K1018,2020 BULKERS
3,2026-09-25T00:00:00,1.43,329408,"[{'date': '2026-09-25T00:00:00', 'shortPercent...",BMG9156K1018,2020 BULKERS
4,2026-09-24T00:00:00,1.09,249963,"[{'date': '2026-09-24T00:00:00', 'shortPercent...",BMG9156K1018,2020 BULKERS


The field activePositions is a list of dictionary items that report detail information on short sellers and the respective holdings, and one needs to further open up this field to have a long table that shows each item as a separate row.

While most records show only one item in the activePositions field (i.e. one position holder's record), there are many cases where more than one position holder's records are presented in the activePositions field, organized in a dictionary. And note that the variable shortPercent actually represents the aggregated shortPercent reported by each position holder in that row.

For instance, below is the content of one activePositions:

[{'date': '2023-03-03',

'shortPercent': 0.68,

'shares': 153129,

'positionHolder': 'Voleon Capital Management LP'},

{'date': '2023-03-07',

'shortPercent': 0.89,

'shares': 198486,

'positionHolder': 'WorldQuant LLC'}]

As some of the records report a complete empty activePositions field, I first remove these records from the entire sample and then parse out the position holder information from this field, and then create a new dataframe that contains all the relevant fields.

In [4]:
# parse out the positionHolder info from the activePositions column
dateLst, holderLst, shortPctLst = [], [], []
shareLst, isinLst, issuerLst = [], [], []

for i in range(dfValid.shape[0]):
    thisIsin = dfValid.iloc[i].isin.__self__.to_dict()['isin']
    thisIssuer = dfValid.iloc[i].issuerName
    for pos in dfValid.iloc[i].activePositions:
        holderLst.append(pos['positionHolder'])
        shortPctLst.append(pos['shortPercent'])
        dateLst.append(pos['date'])
        shareLst.append(pos['shares'])
        isinLst.append(thisIsin)
        issuerLst.append(thisIssuer)

longDf = pd.DataFrame({
    'position_date': dateLst, 'isin': isinLst, 'issuer': issuerLst,
    'position_holder': holderLst, 'net_short_position': shortPctLst,
    'NumPositions': shareLst,
})

Last step is to remove the redundant rows from longDf: the reason is that the json data reports aggregate value by date per row, with each row possibly containing more than one position_holder.

When at least one of the position holders change position, it will lead to update in the aggregate short position, even though the rest of the position holders in that row might not report any changes. With that background in mind, when expanding the position_holders one by one, the ones that do not change positions will get repopulated day after day and therefore appear in duplicate records.

In [5]:
# the API reports one row per date with possibly several position
# holders, and holders who didn't change position get repeated on every
# subsequent date - drop those duplicates
longDf = longDf.drop_duplicates()
longDf.shape

(3537, 6)

And the final output for Norway looks like the following:

In [6]:
#| column: page
longDf.sort_values(['isin', 'position_holder', 'position_date']).head()

,position_date,isin,issuer,position_holder,net_short_position,NumPositions
800,2025-09-30T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.52,477063
799,2025-10-02T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.66,599968
798,2025-10-06T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.70,701581
797,2025-10-10T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.81,807536
796,2025-10-24T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.78,783521
